In [0]:
spark

### ## Datasets Loading and reviewing info

In [0]:
acc=spark.read.csv('/Workspace/Smart_Fraud_Detection/input/accounts.csv',inferSchema=True,header=True)
display(acc.limit(5))

account_id,customer_name,account_type,opening_date,branch,kyc_status,credit_limit
ACC-00001,Rajesh Kapoor,savings,2020-09-21,Mumbai_Main,pending,1000000
ACC-00002,Sunita Deshmukh,NRI,2021-10-14,Delhi_CP,pending,1000000
ACC-00003,Anil Reddy,current,2022-07-03,Bangalore_MG,verified,null
ACC-00004,Meena Iyer,NRI,2021-10-26,Bangalore_MG,verified,100000
ACC-00005,Suresh Patel,current,2016-12-23,Hyderabad_Hitech,expired,50000


In [0]:
transaction=spark.read.csv('/Workspace/Smart_Fraud_Detection/input/transactions.csv',inferSchema=True,header=True)
display(transaction.limit(5))

txn_id,account_id,txn_date,txn_type,amount,merchant,city,is_international
TXN-000001,ACC-00056,2026-01-09,debit,915931.22,null,Singapore,yes
TXN-000002,ACC-00025,2026-01-11,transfer,23892.36,BookMyShow,Bangalore,no
TXN-000003,ACC-00029,2026-02-03,credit,15277.03,Swiggy,New York,yes
TXN-000004,ACC-00007,2026-03-05,withdrawal,5498.74,ATM,Delhi,no
TXN-000005,ACC-00029,2026-02-23,debit,919.78,Flipkart,Chennai,no


In [0]:
fraud=spark.read.csv('/Workspace/Smart_Fraud_Detection/input/known_fraud_accounts.csv',inferSchema=True,header=True)
display(fraud.limit(5))

account_id,fraud_type,flagged_date
ACC-00021,card_cloning,2026-01-14
ACC-00029,money_laundering,2026-01-13
ACC-00031,phishing,2026-03-08
ACC-00009,identity_theft,2026-01-18
ACC-00017,account_takeover,2026-03-08


### ## <b>Bronze Layer :</b> consist of craeting table inside of schema(acts as folder inside a database which consists of tables) and checking for null values and verifying schemas

In [0]:
%sql
create schema if not exists fraud_pipeline ;

In [0]:
acc.write.format('delta').mode('overwrite').saveAsTable('fraud_pipeline.bronze_acc')

In [0]:
transaction.write.format('delta').mode('overwrite').saveAsTable('fraud_pipeline.bronze_transaction')

In [0]:
fraud.write.format('delta').mode('overwrite').saveAsTable('fraud_pipeline.bronze_fraud')

In [0]:
%sql
show tables in fraud_pipeline;

database,tableName,isTemporary
fraud_pipeline,bronze_acc,false
fraud_pipeline,bronze_fraud,false
fraud_pipeline,bronze_transaction,false
fraud_pipeline,silver_df,false


In [0]:
bronze_acc=spark.table('fraud_pipeline.bronze_acc')
bronze_transaction=spark.table('fraud_pipeline.bronze_transaction')
bronze_fraud=spark.table('fraud_pipeline.bronze_fraud')

In [0]:
display(bronze_acc.limit(5))
display(bronze_transaction.limit(5))
display(bronze_fraud.limit(5))

account_id,customer_name,account_type,opening_date,branch,kyc_status,credit_limit
ACC-00001,Rajesh Kapoor,savings,2020-09-21,Mumbai_Main,pending,1000000
ACC-00002,Sunita Deshmukh,NRI,2021-10-14,Delhi_CP,pending,1000000
ACC-00003,Anil Reddy,current,2022-07-03,Bangalore_MG,verified,null
ACC-00004,Meena Iyer,NRI,2021-10-26,Bangalore_MG,verified,100000
ACC-00005,Suresh Patel,current,2016-12-23,Hyderabad_Hitech,expired,50000


txn_id,account_id,txn_date,txn_type,amount,merchant,city,is_international
TXN-000001,ACC-00056,2026-01-09,debit,915931.22,null,Singapore,yes
TXN-000002,ACC-00025,2026-01-11,transfer,23892.36,BookMyShow,Bangalore,no
TXN-000003,ACC-00029,2026-02-03,credit,15277.03,Swiggy,New York,yes
TXN-000004,ACC-00007,2026-03-05,withdrawal,5498.74,ATM,Delhi,no
TXN-000005,ACC-00029,2026-02-23,debit,919.78,Flipkart,Chennai,no


account_id,fraud_type,flagged_date
ACC-00021,card_cloning,2026-01-14
ACC-00029,money_laundering,2026-01-13
ACC-00031,phishing,2026-03-08
ACC-00009,identity_theft,2026-01-18
ACC-00017,account_takeover,2026-03-08


In [0]:
print("Accounts:", bronze_acc.count())
print("Transactions:", bronze_transaction.count())
print("Fraud:", bronze_fraud.count())

Accounts: 50
Transactions: 200
Fraud: 10


In [0]:
from pyspark.sql.functions import col
nullc={}
for c in bronze_acc.columns:
  nullc[c]=bronze_acc.filter(col(c).isNull()).count()
print(nullc)

{'account_id': 0, 'customer_name': 0, 'account_type': 0, 'opening_date': 0, 'branch': 0, 'kyc_status': 0, 'credit_limit': 9}


In [0]:
from pyspark.sql.functions import col
nullc={}
for c in bronze_transaction.columns:
  nullc[c]=bronze_transaction.filter(col(c).isNull()).count()
print(nullc)

{'txn_id': 0, 'account_id': 0, 'txn_date': 0, 'txn_type': 0, 'amount': 0, 'merchant': 7, 'city': 0, 'is_international': 0}


In [0]:
from pyspark.sql.functions import col
nullc={}
for c in bronze_fraud.columns:
  nullc[c]=bronze_fraud.filter(col(c).isNull()).count()
print(nullc)

{'account_id': 0, 'fraud_type': 0, 'flagged_date': 0}


In [0]:


bronze_acc.write.format("delta").mode("overwrite").save("/Workspace/Smart_Fraud_Detection/bronze/accounts")
bronze_transaction.write.format("delta").mode("overwrite").save("/Workspace/Smart_Fraud_Detection/bronze/transactions")
bronze_fraud.write.format("delta").mode("overwrite").save("/Workspace/Smart_Fraud_Detection/bronze/fraud")

### ## <b>Silver layer :</b> removing duplicate values filling null with appropriate values , cast data types if required and join tables

In [0]:
accounts_clean = bronze_acc.dropDuplicates()
transactions_clean = bronze_transaction.dropDuplicates()
fraud_clean = bronze_fraud.dropDuplicates()
print(accounts_clean.count())
print(transactions_clean.count())
print(fraud_clean.count())

50
200
10


In [0]:
accounts_clean = bronze_acc.fillna({
    "credit_limit": 0
})
transactions_clean = bronze_transaction.fillna({
    "merchant": "Unknown"
})

In [0]:
accounts_clean.printSchema()
transactions_clean.printSchema()
fraud_clean.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- branch: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- credit_limit: integer (nullable = false)

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant: string (nullable = false)
 |-- city: string (nullable = true)
 |-- is_international: string (nullable = true)

root
 |-- account_id: string (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- flagged_date: date (nullable = true)



In [0]:
silver_df = transactions_clean.join(
    accounts_clean,
    on="account_id",
    how="inner"
)
display(silver_df)

account_id,txn_id,txn_date,txn_type,amount,merchant,city,is_international,customer_name,account_type,opening_date,branch,kyc_status,credit_limit
ACC-00025,TXN-000002,2026-01-11,transfer,23892.36,BookMyShow,Bangalore,no,Hemant Jain,NRI,2023-02-17,Hyderabad_Hitech,verified,500000
ACC-00029,TXN-000003,2026-02-03,credit,15277.03,Swiggy,New York,yes,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000
ACC-00007,TXN-000004,2026-03-05,withdrawal,5498.74,ATM,Delhi,no,Prakash Rao,NRI,2016-10-04,Pune_FC,expired,500000
ACC-00029,TXN-000005,2026-02-23,debit,919.78,Flipkart,Chennai,no,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000
ACC-00047,TXN-000006,2026-02-09,debit,22417.98,Swiggy,Mumbai,no,Pawan Arora,savings,2024-02-02,Hyderabad_Hitech,pending,100000
ACC-00003,TXN-000007,2026-02-22,transfer,3832.61,BigBasket,Singapore,yes,Anil Reddy,current,2022-07-03,Bangalore_MG,verified,0
ACC-00037,TXN-000008,2026-03-01,withdrawal,11979.58,PhonePe,Delhi,no,Naveen Chadha,salary,2015-11-09,Kolkata_Park,expired,50000
ACC-00035,TXN-000009,2026-01-08,withdrawal,23856.18,Unknown_Merchant,New York,yes,Tarun Bose,savings,2017-10-27,Delhi_CP,expired,500000
ACC-00008,TXN-000010,2026-03-19,transfer,8369.32,BookMyShow,Singapore,yes,Lata Mishra,salary,2022-11-22,Mumbai_Main,verified,100000
ACC-00004,TXN-000011,2026-03-21,debit,9165.62,Unknown_Merchant,London,yes,Meena Iyer,NRI,2021-10-26,Bangalore_MG,verified,100000


In [0]:
silver_df.write.mode("overwrite").saveAsTable("fraud_pipeline.silver_df")

In [0]:
%sql
SHOW TABLES IN fraud_pipeline;

database,tableName,isTemporary
fraud_pipeline,bronze_acc,false
fraud_pipeline,bronze_fraud,false
fraud_pipeline,bronze_transaction,false
fraud_pipeline,silver_df,false


In [0]:
silver_df.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant: string (nullable = false)
 |-- city: string (nullable = true)
 |-- is_international: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- branch: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- credit_limit: integer (nullable = false)



In [0]:
%sql
SHOW TABLES IN fraud_pipeline;

database,tableName,isTemporary
fraud_pipeline,bronze_acc,false
fraud_pipeline,bronze_fraud,false
fraud_pipeline,bronze_transaction,false
fraud_pipeline,silver_df,false


In [0]:
fraud_df = spark.table("fraud_pipeline.bronze_fraud")

In [0]:

silver_df.write.format("delta").mode("overwrite").save("/Workspace/Smart_Fraud_Detection/silver/silver_transactions")

### <b>Gold layer :</b> consist of buisness insights 

In [0]:
gold_df = silver_df.join(
    fraud_df,
    on="account_id",
    how="left"
)
display(gold_df)

account_id,txn_id,txn_date,txn_type,amount,merchant,city,is_international,customer_name,account_type,opening_date,branch,kyc_status,credit_limit,fraud_type,flagged_date
ACC-00025,TXN-000002,2026-01-11,transfer,23892.36,BookMyShow,Bangalore,no,Hemant Jain,NRI,2023-02-17,Hyderabad_Hitech,verified,500000,null,null
ACC-00029,TXN-000003,2026-02-03,credit,15277.03,Swiggy,New York,yes,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13
ACC-00007,TXN-000004,2026-03-05,withdrawal,5498.74,ATM,Delhi,no,Prakash Rao,NRI,2016-10-04,Pune_FC,expired,500000,null,null
ACC-00029,TXN-000005,2026-02-23,debit,919.78,Flipkart,Chennai,no,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13
ACC-00047,TXN-000006,2026-02-09,debit,22417.98,Swiggy,Mumbai,no,Pawan Arora,savings,2024-02-02,Hyderabad_Hitech,pending,100000,null,null
ACC-00003,TXN-000007,2026-02-22,transfer,3832.61,BigBasket,Singapore,yes,Anil Reddy,current,2022-07-03,Bangalore_MG,verified,0,null,null
ACC-00037,TXN-000008,2026-03-01,withdrawal,11979.58,PhonePe,Delhi,no,Naveen Chadha,salary,2015-11-09,Kolkata_Park,expired,50000,null,null
ACC-00035,TXN-000009,2026-01-08,withdrawal,23856.18,Unknown_Merchant,New York,yes,Tarun Bose,savings,2017-10-27,Delhi_CP,expired,500000,null,null
ACC-00008,TXN-000010,2026-03-19,transfer,8369.32,BookMyShow,Singapore,yes,Lata Mishra,salary,2022-11-22,Mumbai_Main,verified,100000,null,null
ACC-00004,TXN-000011,2026-03-21,debit,9165.62,Unknown_Merchant,London,yes,Meena Iyer,NRI,2021-10-26,Bangalore_MG,verified,100000,null,null


In [0]:
gold_df.printSchema()
print(gold_df.columns)

root
 |-- account_id: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant: string (nullable = false)
 |-- city: string (nullable = true)
 |-- is_international: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- branch: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- credit_limit: integer (nullable = false)
 |-- fraud_type: string (nullable = true)
 |-- flagged_date: date (nullable = true)

['account_id', 'txn_id', 'txn_date', 'txn_type', 'amount', 'merchant', 'city', 'is_international', 'customer_name', 'account_type', 'opening_date', 'branch', 'kyc_status', 'credit_limit', 'fraud_type', 'flagged_date']


In [0]:
%sql
SHOW TABLES IN fraud_pipeline;

database,tableName,isTemporary
fraud_pipeline,bronze_acc,false
fraud_pipeline,bronze_fraud,false
fraud_pipeline,bronze_transaction,false
fraud_pipeline,silver_df,false


<p> Buisness Insights </p>

In [0]:
gold_df.count()

192

In [0]:
gold_df.groupBy("fraud_type").count().display()

fraud_type,count
card_cloning,6
money_laundering,9
account_takeover,2
phishing,4
null,168
identity_theft,3


In [0]:
gold_df.filter(col("fraud_type").isNotNull()).groupBy("branch").count().display()

branch,count
Kolkata_Park,4
Bangalore_MG,5
Pune_FC,11
Chennai_T_Nagar,4


In [0]:
gold_df.createOrReplaceTempView("temp")

### Creating a new column as fraud status so it will be easy for us to differentiate

In [0]:
gold_df = spark.sql("""
SELECT *,
       CASE
            WHEN fraud_type IS NOT NULL THEN 'Fraud'
            ELSE 'Normal'
       END AS fraud_status
FROM temp
""")

In [0]:
display(gold_df.limit(5))

account_id,txn_id,txn_date,txn_type,amount,merchant,city,is_international,customer_name,account_type,opening_date,branch,kyc_status,credit_limit,fraud_type,flagged_date,fraud_status
ACC-00025,TXN-000002,2026-01-11,transfer,23892.36,BookMyShow,Bangalore,no,Hemant Jain,NRI,2023-02-17,Hyderabad_Hitech,verified,500000,null,null,Normal
ACC-00029,TXN-000003,2026-02-03,credit,15277.03,Swiggy,New York,yes,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13,Fraud
ACC-00007,TXN-000004,2026-03-05,withdrawal,5498.74,ATM,Delhi,no,Prakash Rao,NRI,2016-10-04,Pune_FC,expired,500000,null,null,Normal
ACC-00029,TXN-000005,2026-02-23,debit,919.78,Flipkart,Chennai,no,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13,Fraud
ACC-00047,TXN-000006,2026-02-09,debit,22417.98,Swiggy,Mumbai,no,Pawan Arora,savings,2024-02-02,Hyderabad_Hitech,pending,100000,null,null,Normal


In [0]:
gold_df.write.mode("overwrite").saveAsTable("gold_fin")

In [0]:
%sql
select count(fraud_type) as Total_Fraud from gold_fin where fraud_status = 'Fraud'

Total_Fraud
24


In [0]:
%sql
SELECT COUNT(*) AS Total_Transactions
FROM gold_fin;

Total_Transactions
192


In [0]:
%sql
SELECT fraud_status,
       COUNT(*) AS Total
FROM gold_fin
GROUP BY fraud_status;

fraud_status,Total
Normal,168
Fraud,24


In [0]:
%sql
SELECT branch,
       COUNT(*) AS Fraud_Count
FROM gold_fin
WHERE fraud_status = 'Fraud'
GROUP BY branch
ORDER BY Fraud_Count DESC;

branch,Fraud_Count
Pune_FC,11
Bangalore_MG,5
Chennai_T_Nagar,4
Kolkata_Park,4


In [0]:
%sql
SELECT account_type,
       COUNT(*) AS Fraud_Count
FROM gold_fin
WHERE fraud_status = 'Fraud'
GROUP BY account_type;

account_type,Fraud_Count
salary,6
NRI,5
current,11
savings,2


In [0]:
%sql
SELECT SUM(amount) AS Total_Fraud_Amount
FROM gold_fin
WHERE fraud_status = 'Fraud';

Total_Fraud_Amount
2885252.5499999993


In [0]:
%sql
SELECT account_id,
       customer_name,
       amount,
       fraud_type
FROM gold_fin
WHERE fraud_status = 'Fraud'
ORDER BY amount DESC;

account_id,customer_name,amount,fraud_type
ACC-00038,Sushma Thakur,1034145.96,card_cloning
ACC-00029,Tanuja Nair,807733.04,money_laundering
ACC-00038,Sushma Thakur,798127.09,card_cloning
ACC-00017,Ravi Pandey,24357.76,account_takeover
ACC-00012,Rekha Bansal,21574.61,identity_theft
ACC-00029,Tanuja Nair,20876.82,money_laundering
ACC-00029,Tanuja Nair,20747.52,money_laundering
ACC-00031,Akash Dhawan,19901.51,phishing
ACC-00038,Sushma Thakur,19786.75,card_cloning
ACC-00009,Venkat Naidu,15818.93,identity_theft


In [0]:
%sql
SELECT txn_type,
       COUNT(*) AS Fraud_Count
FROM gold_fin
WHERE fraud_status = 'Fraud'
GROUP BY txn_type;

txn_type,Fraud_Count
credit,5
debit,4
payment,4
withdrawal,6
transfer,5


In [0]:
%sql
SELECT is_international,
       COUNT(*) AS Fraud_Count
FROM gold_fin
WHERE fraud_status = 'Fraud'
GROUP BY is_international;

is_international,Fraud_Count
yes,14
no,10


In [0]:

gold_df.write.format("delta").mode("overwrite").save("/Workspace/Smart_Fraud_Detection/gold/gold_transactions")

### saving in csv for consumer to have easy readability 

In [0]:
gold_df.write.format("csv").mode("overwrite").save("/Workspace/Smart_Fraud_Detection/output/cvs/output")

In [0]:
final=spark.read.csv("/Workspace/Smart_Fraud_Detection/output/cvs/output",header=True,inferSchema=True)
display(final.limit(10))

ACC-00025,TXN-000002,2026-01-11,transfer,23892.36,BookMyShow,Bangalore,no,Hemant Jain,NRI,2023-02-17,Hyderabad_Hitech,verified,500000,_c14,_c15,Normal
ACC-00029,TXN-000003,2026-02-03,credit,15277.03,Swiggy,New York,yes,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13,Fraud
ACC-00007,TXN-000004,2026-03-05,withdrawal,5498.74,ATM,Delhi,no,Prakash Rao,NRI,2016-10-04,Pune_FC,expired,500000,null,null,Normal
ACC-00029,TXN-000005,2026-02-23,debit,919.78,Flipkart,Chennai,no,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13,Fraud
ACC-00047,TXN-000006,2026-02-09,debit,22417.98,Swiggy,Mumbai,no,Pawan Arora,savings,2024-02-02,Hyderabad_Hitech,pending,100000,null,null,Normal
ACC-00003,TXN-000007,2026-02-22,transfer,3832.61,BigBasket,Singapore,yes,Anil Reddy,current,2022-07-03,Bangalore_MG,verified,0,null,null,Normal
ACC-00037,TXN-000008,2026-03-01,withdrawal,11979.58,PhonePe,Delhi,no,Naveen Chadha,salary,2015-11-09,Kolkata_Park,expired,50000,null,null,Normal
ACC-00035,TXN-000009,2026-01-08,withdrawal,23856.18,Unknown_Merchant,New York,yes,Tarun Bose,savings,2017-10-27,Delhi_CP,expired,500000,null,null,Normal
ACC-00008,TXN-000010,2026-03-19,transfer,8369.32,BookMyShow,Singapore,yes,Lata Mishra,salary,2022-11-22,Mumbai_Main,verified,100000,null,null,Normal
ACC-00004,TXN-000011,2026-03-21,debit,9165.62,Unknown_Merchant,London,yes,Meena Iyer,NRI,2021-10-26,Bangalore_MG,verified,100000,null,null,Normal
ACC-00006,TXN-000012,2026-02-27,transfer,16212.42,Swiggy,Tokyo,yes,Kavita Joshi,NRI,2020-09-04,Kolkata_Park,verified,200000,null,null,Normal
